# T-03: Speaker Identification and Voice-Activity Detection

## Narrow project question

In three continuous 300-second excerpts from the AMI Meeting Corpus, how can a simple inference-only pipeline combining pretrained Silero VAD and ECAPA-TDNN detect speech and identify the active speaker among 12 enrolled speakers?

The project uses closed-set speaker identification. Every single-speaker interval considered for speaker accuracy must belong to one of the 12 enrolled speakers.

In [ ]:
%pip install -q speechbrain==1.1.0 silero-vad==6.2.1 soundfile==0.14.0

In [ ]:
from pathlib import Path
import urllib.request

import soundfile as sf
import torch
import torchaudio

from silero_vad import get_speech_timestamps, load_silero_vad
from speechbrain.inference.classifiers import EncoderClassifier

## 1. Download the AMI meeting data

This project uses three meetings from the AMI Meeting Corpus: `ES2002d`, `ES2008d`, and `ES2014d`. For each meeting, the mono Mix-Headset recording and its RTTM speaker annotations are downloaded.

The audio recordings contain natural meeting speech, silence, and overlapping speech. The RTTM files provide the start time, duration, and speaker identity of each annotated speech segment. Across the three selected meetings, there are 12 meeting-speaker identities.

In [ ]:
MEETING_IDS = [
    "ES2002d",
    "ES2008d",
    "ES2014d"]

DATA_DIR = Path("/content/ami_subset")
DATA_DIR.mkdir(parents=True, exist_ok=True)

meeting_data = {}

for meeting_id in MEETING_IDS:
    audio_path = (
        DATA_DIR
        / f"{meeting_id}.Mix-Headset.wav"
    )

    rttm_path = (
        DATA_DIR
        / f"{meeting_id}.rttm"
    )

    audio_url = (
        f"https://groups.inf.ed.ac.uk/ami/"
        f"AMICorpusMirror/amicorpus/"
        f"{meeting_id}/audio/"
        f"{meeting_id}.Mix-Headset.wav"
    )

    rttm_url = (
        f"https://raw.githubusercontent.com/"
        f"BUTSpeechFIT/AMI-diarization-setup/"
        f"main/only_words/rttms/train/"
        f"{meeting_id}.rttm"
    )

    if not audio_path.exists():
        print(
            f"Downloading audio for {meeting_id}..."
        )
        urllib.request.urlretrieve(
            audio_url,
            audio_path,
        )

    if not rttm_path.exists():
        print(
            f"Downloading RTTM for {meeting_id}..."
        )
        urllib.request.urlretrieve(
            rttm_url,
            rttm_path,
        )

    meeting_data[meeting_id] = {
        "audio_path": audio_path,
        "rttm_path": rttm_path,
    }

    audio_size_mb = (
        audio_path.stat().st_size
        / (1024 ** 2)
    )

    print(
        f"{meeting_id}: ready, "
        f"audio size {audio_size_mb:.2f} MB"
    )

print(
    f"\nPrepared {len(meeting_data)} meetings."
)

ES2002d: ready, audio size 80.08 MB
ES2008d: ready, audio size 80.13 MB
ES2014d: ready, audio size 88.85 MB

Prepared 3 meetings.


## 2. Load the RTTM annotations

Each RTTM annotation specifies the meeting, speaker identity, speech start time, and speech duration. These annotations are used to select clean enrollment segments and to create the reference timeline for evaluation.

In [ ]:
def load_rttm(rttm_path, meeting_id):
    segments = []

    with open(rttm_path, "r", encoding="utf-8") as file:
        for line in file:
            fields = line.strip().split()

            if len(fields) < 8 or fields[0] != "SPEAKER":
                continue

            start = float(fields[3])
            duration = float(fields[4])
            speaker = fields[7]

            segments.append(
                {
                    "meeting": meeting_id,
                    "speaker": speaker,
                    "speaker_label": f"{meeting_id}/{speaker}",
                    "start": start,
                    "end": start + duration,
                    "duration": duration,
                }
            )

    return sorted(segments, key=lambda segment: segment["start"])


annotations_by_meeting = {}

for meeting_id in MEETING_IDS:
    rttm_path = DATA_DIR / f"{meeting_id}.rttm"

    segments = load_rttm(rttm_path, meeting_id)
    annotations_by_meeting[meeting_id] = segments

    speakers = sorted({segment["speaker"] for segment in segments})

    print(f"{meeting_id}:")
    print(f"  Annotated segments: {len(segments)}")
    print(f"  Speakers ({len(speakers)}): {speakers}")


total_segments = sum(
    len(segments)
    for segments in annotations_by_meeting.values()
)

total_speakers = sum(
    len({segment["speaker"] for segment in segments})
    for segments in annotations_by_meeting.values()
)

print()
print(f"Total annotated segments: {total_segments}")
print(f"Total meeting-speaker identities: {total_speakers}")

ES2002d:
  Annotated segments: 774
  Speakers (4): ['FEE005', 'MEE006', 'MEE007', 'MEE008']
ES2008d:
  Annotated segments: 757
  Speakers (4): ['FEE029', 'FEE030', 'FEE032', 'MEE031']
ES2014d:
  Annotated segments: 695
  Speakers (4): ['FEE055', 'MEE053', 'MEE054', 'MEE056']

Total annotated segments: 2226
Total meeting-speaker identities: 12


## 3. Select clean enrollment segments

Five clean enrollment segments are selected for each speaker. An enrollment segment must be between 0.8 and 10 seconds long, must not overlap with speech from another speaker, and must end before the continuous evaluation interval begins.

The evaluation interval starts at 2100 seconds, so enrollment and evaluation data are temporally separated. The enrollment segments are used only to create speaker centroids. Final speaker-identification evaluation will be performed on the actual speech intervals detected by Silero VAD in continuous audio.

In [ ]:
MIN_ENROLLMENT_DURATION = 0.8
MAX_ENROLLMENT_DURATION = 10.0
ENROLLMENT_SEGMENTS_PER_SPEAKER = 5

EVALUATION_START = 2100.0
EVALUATION_END = 2400.0


def overlaps_another_speaker(target_segment, all_segments):
    for other_segment in all_segments:
        if other_segment["speaker"] == target_segment["speaker"]:
            continue

        overlap_start = max(
            target_segment["start"],
            other_segment["start"],
        )
        overlap_end = min(
            target_segment["end"],
            other_segment["end"],
        )

        if overlap_end > overlap_start:
            return True

    return False


clean_candidates_by_speaker = {}
enrollment_segments = []

for meeting_id in MEETING_IDS:
    meeting_segments = annotations_by_meeting[meeting_id]
    meeting_speakers = sorted(
        {segment["speaker"] for segment in meeting_segments}
    )

    print(f"{meeting_id}:")

    for speaker in meeting_speakers:
        speaker_label = f"{meeting_id}/{speaker}"

        candidates = [
            segment
            for segment in meeting_segments
            if segment["speaker"] == speaker
            and MIN_ENROLLMENT_DURATION
            <= segment["duration"]
            <= MAX_ENROLLMENT_DURATION
            and segment["end"] <= EVALUATION_START
            and not overlaps_another_speaker(
                segment,
                meeting_segments,
            )
        ]

        candidates = sorted(
            candidates,
            key=lambda segment: segment["start"],
        )

        clean_candidates_by_speaker[speaker_label] = candidates

        if len(candidates) < ENROLLMENT_SEGMENTS_PER_SPEAKER:
            raise ValueError(
                f"Not enough clean enrollment segments for "
                f"{speaker_label}: {len(candidates)} found."
            )

        selected_segments = candidates[
            :ENROLLMENT_SEGMENTS_PER_SPEAKER
        ]
        enrollment_segments.extend(selected_segments)

        print(
            f"  {speaker}: "
            f"{len(candidates)} clean candidates, "
            f"{len(selected_segments)} selected"
        )


enrollment_speakers = sorted(
    {segment["speaker_label"] for segment in enrollment_segments}
)

enrollment_durations = [
    segment["duration"] for segment in enrollment_segments
]

latest_enrollment_end = max(
    segment["end"] for segment in enrollment_segments
)

print()
print(f"Total enrollment segments: {len(enrollment_segments)}")
print(f"Total enrolled speakers: {len(enrollment_speakers)}")
print(
    "Enrollment duration range: "
    f"{min(enrollment_durations):.2f} to "
    f"{max(enrollment_durations):.2f} seconds"
)
print(
    f"Latest enrollment end time: "
    f"{latest_enrollment_end:.2f} seconds"
)
print(
    f"Evaluation interval: "
    f"{EVALUATION_START:.0f} to "
    f"{EVALUATION_END:.0f} seconds"
)

ES2002d:
  FEE005: 24 clean candidates, 5 selected
  MEE006: 10 clean candidates, 5 selected
  MEE007: 14 clean candidates, 5 selected
  MEE008: 20 clean candidates, 5 selected
ES2008d:
  FEE029: 42 clean candidates, 5 selected
  FEE030: 38 clean candidates, 5 selected
  FEE032: 21 clean candidates, 5 selected
  MEE031: 12 clean candidates, 5 selected
ES2014d:
  FEE055: 16 clean candidates, 5 selected
  MEE053: 27 clean candidates, 5 selected
  MEE054: 8 clean candidates, 5 selected
  MEE056: 33 clean candidates, 5 selected

Total enrollment segments: 60
Total enrolled speakers: 12
Enrollment duration range: 0.82 to 9.92 seconds
Latest enrollment end time: 1826.23 seconds
Evaluation interval: 2100 to 2400 seconds


## 4. Inspect recordings and define audio extraction

The sample rate, channel count, and duration of each recording are checked before inference. To reduce memory usage, only the required enrollment or evaluation interval is read from each recording instead of loading the complete meeting into memory.

In [ ]:
TARGET_SAMPLE_RATE = 16000
audio_paths = {}

for meeting_id in MEETING_IDS:
    audio_path = DATA_DIR / f"{meeting_id}.Mix-Headset.wav"
    audio_info = sf.info(str(audio_path))

    if audio_info.duration < EVALUATION_END:
        raise ValueError(
            f"{meeting_id} is shorter than the selected "
            f"evaluation interval."
        )

    audio_paths[meeting_id] = audio_path

    print(f"{meeting_id}:")
    print(f"  Sample rate: {audio_info.samplerate} Hz")
    print(f"  Channels: {audio_info.channels}")
    print(f"  Duration: {audio_info.duration:.2f} seconds")
    print(f"  Duration: {audio_info.duration / 60:.2f} minutes")


def read_audio_segment(meeting_id, start, end):
    audio_path = audio_paths[meeting_id]

    with sf.SoundFile(str(audio_path), mode="r") as audio_file:
        source_sample_rate = audio_file.samplerate

        start_frame = round(start * source_sample_rate)
        number_of_frames = round(
            (end - start) * source_sample_rate
        )

        audio_file.seek(start_frame)

        audio_array = audio_file.read(
            frames=number_of_frames,
            dtype="float32",
            always_2d=True,
        )

    waveform = torch.from_numpy(audio_array).mean(dim=1)

    if source_sample_rate != TARGET_SAMPLE_RATE:
        waveform = torchaudio.functional.resample(
            waveform,
            source_sample_rate,
            TARGET_SAMPLE_RATE,
        )

    return waveform.contiguous()

ES2002d:
  Sample rate: 16000 Hz
  Channels: 1
  Duration: 2624.17 seconds
  Duration: 43.74 minutes
ES2008d:
  Sample rate: 16000 Hz
  Channels: 1
  Duration: 2625.82 seconds
  Duration: 43.76 minutes
ES2014d:
  Sample rate: 16000 Hz
  Channels: 1
  Duration: 2911.36 seconds
  Duration: 48.52 minutes


## 5. Load the pretrained baseline models

The pretrained Silero VAD model is used to detect speech intervals in continuous meeting audio. SpeechBrain's pretrained ECAPA-TDNN model is used to convert each speech interval into a speaker embedding.

Both models are used only for inference on CPU. No model training or fine-tuning is performed.

In [ ]:
DEVICE = "cpu"

vad_model = load_silero_vad()

speaker_model = EncoderClassifier.from_hparams(
    source="speechbrain/spkrec-ecapa-voxceleb",
    savedir="/content/pretrained_models/ecapa_voxceleb",
    run_opts={"device": DEVICE},
)

print("Silero VAD loaded successfully.")
print("SpeechBrain ECAPA-TDNN loaded successfully.")

INFO:speechbrain.utils.fetching:Fetch hyperparams.yaml: Using symlink found at '/content/pretrained_models/ecapa_voxceleb/hyperparams.yaml'
INFO:speechbrain.utils.fetching:Fetch embedding_model.ckpt: Using symlink found at '/content/pretrained_models/ecapa_voxceleb/embedding_model.ckpt'
INFO:speechbrain.utils.fetching:Fetch mean_var_norm_emb.ckpt: Using symlink found at '/content/pretrained_models/ecapa_voxceleb/mean_var_norm_emb.ckpt'
INFO:speechbrain.utils.fetching:Fetch classifier.ckpt: Using symlink found at '/content/pretrained_models/ecapa_voxceleb/classifier.ckpt'
INFO:speechbrain.utils.fetching:Fetch label_encoder.txt: Using symlink found at '/content/pretrained_models/ecapa_voxceleb/label_encoder.ckpt'
INFO:speechbrain.utils.parameter_transfer:Loading pretrained files for: embedding_model, mean_var_norm_emb, classifier, label_encoder


Silero VAD loaded successfully.
SpeechBrain ECAPA-TDNN loaded successfully.


## 6. Create enrolled speaker centroids

ECAPA-TDNN converts each enrollment segment into a 192-dimensional speaker embedding. The five normalized embeddings belonging to each enrolled speaker are averaged and normalized again to create one speaker centroid.

Speaker identification will later compare each VAD-detected speech interval with these 12 centroids using cosine similarity.

In [ ]:
@torch.inference_mode()
def extract_speaker_embedding(waveform):
    if waveform.numel() == 0:
        raise ValueError("Cannot extract an embedding from empty audio.")

    waveform = waveform.to(DEVICE).unsqueeze(0)

    embedding = speaker_model.encode_batch(
        waveform
    ).squeeze()

    embedding = embedding.cpu()
    embedding = embedding / embedding.norm(p=2)

    return embedding


enrollment_embeddings = {}

for speaker_label in enrollment_speakers:
    speaker_segments = [
        segment
        for segment in enrollment_segments
        if segment["speaker_label"] == speaker_label
    ]

    speaker_embedding_list = []

    print(f"Processing {speaker_label}...")

    for segment in speaker_segments:
        waveform = read_audio_segment(
            meeting_id=segment["meeting"],
            start=segment["start"],
            end=segment["end"],
        )

        embedding = extract_speaker_embedding(waveform)
        speaker_embedding_list.append(embedding)

    enrollment_embeddings[speaker_label] = speaker_embedding_list


speaker_centroids = {}

for speaker_label in enrollment_speakers:
    centroid = torch.stack(
        enrollment_embeddings[speaker_label]
    ).mean(dim=0)

    centroid = centroid / centroid.norm(p=2)
    speaker_centroids[speaker_label] = centroid

    print(
        f"{speaker_label}: "
        f"{len(enrollment_embeddings[speaker_label])} segments, "
        f"centroid shape {tuple(centroid.shape)}, "
        f"norm {centroid.norm(p=2).item():.4f}"
    )


print()
print(f"Created centroids for {len(speaker_centroids)} speakers.")

Processing ES2002d/FEE005...
Processing ES2002d/MEE006...
Processing ES2002d/MEE007...
Processing ES2002d/MEE008...
Processing ES2008d/FEE029...
Processing ES2008d/FEE030...
Processing ES2008d/FEE032...
Processing ES2008d/MEE031...
Processing ES2014d/FEE055...
Processing ES2014d/MEE053...
Processing ES2014d/MEE054...
Processing ES2014d/MEE056...
ES2002d/FEE005: 5 segments, centroid shape (192,), norm 1.0000
ES2002d/MEE006: 5 segments, centroid shape (192,), norm 1.0000
ES2002d/MEE007: 5 segments, centroid shape (192,), norm 1.0000
ES2002d/MEE008: 5 segments, centroid shape (192,), norm 1.0000
ES2008d/FEE029: 5 segments, centroid shape (192,), norm 1.0000
ES2008d/FEE030: 5 segments, centroid shape (192,), norm 1.0000
ES2008d/FEE032: 5 segments, centroid shape (192,), norm 1.0000
ES2008d/MEE031: 5 segments, centroid shape (192,), norm 1.0000
ES2014d/FEE055: 5 segments, centroid shape (192,), norm 1.0000
ES2014d/MEE053: 5 segments, centroid shape (192,), norm 1.0000
ES2014d/MEE054: 5 segm

## 7. Verify the speaker-identification component

As a Phase 1 test, the speaker-identification component is applied to one clean segment that was not used for enrollment. Its ECAPA-TDNN embedding is compared with all 12 enrolled centroids using cosine similarity.

This isolated check only verifies that the component runs correctly. In Phase 2, speaker identification will instead be applied to the intervals produced by Silero VAD from continuous audio.

In [ ]:
def identify_speaker(waveform):
    embedding = extract_speaker_embedding(waveform)

    similarity_scores = {
        speaker_label: torch.dot(
            embedding,
            centroid,
        ).item()
        for speaker_label, centroid
        in speaker_centroids.items()
    }

    predicted_speaker = max(
        similarity_scores,
        key=similarity_scores.get,
    )

    return predicted_speaker, similarity_scores


example_speaker = enrollment_speakers[0]

example_segment = clean_candidates_by_speaker[
    example_speaker
][ENROLLMENT_SEGMENTS_PER_SPEAKER]

example_waveform = read_audio_segment(
    meeting_id=example_segment["meeting"],
    start=example_segment["start"],
    end=example_segment["end"],
)

predicted_speaker, similarity_scores = identify_speaker(
    example_waveform
)

sorted_scores = sorted(
    similarity_scores.items(),
    key=lambda item: item[1],
    reverse=True,
)

print("Speaker-identification test:")
print(f"Meeting: {example_segment['meeting']}")
print(f"True speaker: {example_segment['speaker_label']}")
print(f"Predicted speaker: {predicted_speaker}")
print(f"Duration: {example_segment['duration']:.2f} seconds")

print()
print("Cosine similarity scores:")

for speaker_label, score in sorted_scores:
    print(f"  {speaker_label}: {score:.4f}")

Speaker-identification test:
Meeting: ES2002d
True speaker: ES2002d/FEE005
Predicted speaker: ES2002d/FEE005
Duration: 3.11 seconds

Cosine similarity scores:
  ES2002d/FEE005: 0.7257
  ES2014d/FEE055: 0.2303
  ES2014d/MEE056: 0.1795
  ES2008d/FEE030: 0.1695
  ES2002d/MEE006: 0.1236
  ES2008d/FEE029: 0.1146
  ES2014d/MEE054: 0.1008
  ES2002d/MEE007: 0.0701
  ES2008d/MEE031: 0.0548
  ES2014d/MEE053: 0.0413
  ES2002d/MEE008: 0.0381
  ES2008d/FEE032: 0.0053


## 8. Prepare the continuous evaluation excerpts

A continuous 300-second interval from 2100 to 2400 seconds is selected from each meeting. These intervals occur after all selected enrollment segments and contain speech, silence, and overlapping speech.

RTTM annotations are clipped to each evaluation interval and converted to local times between 0 and 300 seconds. They will later provide the reference for VAD evaluation and for determining which speaker or speakers are active during each detected interval.

In [ ]:
EVALUATION_DURATION = EVALUATION_END - EVALUATION_START


def merge_intervals(intervals):
    if not intervals:
        return []

    sorted_intervals = sorted(
        intervals,
        key=lambda interval: interval["start"],
    )

    merged = [sorted_intervals[0].copy()]

    for interval in sorted_intervals[1:]:
        previous = merged[-1]

        if interval["start"] <= previous["end"]:
            previous["end"] = max(
                previous["end"],
                interval["end"],
            )
        else:
            merged.append(interval.copy())

    for interval in merged:
        interval["duration"] = (
            interval["end"] - interval["start"]
        )

    return merged


def calculate_timeline_durations(
    merged_intervals_by_speaker,
):
    events = []

    for speaker_intervals in (
        merged_intervals_by_speaker.values()
    ):
        for interval in speaker_intervals:
            events.append((interval["start"], 1))
            events.append((interval["end"], -1))

    events.sort()

    active_speakers = 0
    previous_time = 0.0
    speech_duration = 0.0
    overlap_duration = 0.0

    for event_time, change in events:
        interval_duration = event_time - previous_time

        if active_speakers >= 1:
            speech_duration += interval_duration

        if active_speakers >= 2:
            overlap_duration += interval_duration

        active_speakers += change
        previous_time = event_time

    return speech_duration, overlap_duration


evaluation_waveforms = {}
reference_segments_by_meeting = {}
reference_intervals_by_speaker = {}
reference_speech_intervals = {}

total_reference_speech = 0.0
total_reference_overlap = 0.0

for meeting_id in MEETING_IDS:
    evaluation_waveform = read_audio_segment(
        meeting_id=meeting_id,
        start=EVALUATION_START,
        end=EVALUATION_END,
    )

    evaluation_waveforms[meeting_id] = evaluation_waveform

    local_segments = []

    for segment in annotations_by_meeting[meeting_id]:
        clipped_start = max(
            segment["start"],
            EVALUATION_START,
        )
        clipped_end = min(
            segment["end"],
            EVALUATION_END,
        )

        if clipped_end <= clipped_start:
            continue

        local_segments.append(
            {
                "meeting": meeting_id,
                "speaker": segment["speaker"],
                "speaker_label": segment["speaker_label"],
                "start": clipped_start - EVALUATION_START,
                "end": clipped_end - EVALUATION_START,
                "duration": clipped_end - clipped_start,
            }
        )

    reference_segments_by_meeting[meeting_id] = (
        local_segments
    )

    meeting_speakers = sorted(
        {
            segment["speaker_label"]
            for segment in local_segments
        }
    )

    merged_by_speaker = {}

    for speaker_label in meeting_speakers:
        speaker_intervals = [
            {
                "start": segment["start"],
                "end": segment["end"],
            }
            for segment in local_segments
            if segment["speaker_label"] == speaker_label
        ]

        merged_by_speaker[speaker_label] = (
            merge_intervals(speaker_intervals)
        )

    reference_intervals_by_speaker[meeting_id] = (
        merged_by_speaker
    )

    all_speaker_intervals = [
        interval
        for speaker_intervals in merged_by_speaker.values()
        for interval in speaker_intervals
    ]

    meeting_speech_intervals = merge_intervals(
        all_speaker_intervals
    )

    reference_speech_intervals[meeting_id] = (
        meeting_speech_intervals
    )

    speech_duration, overlap_duration = (
        calculate_timeline_durations(merged_by_speaker)
    )

    silence_duration = (
        EVALUATION_DURATION - speech_duration
    )

    total_reference_speech += speech_duration
    total_reference_overlap += overlap_duration

    print(f"{meeting_id}:")
    print(
        f"  Waveform shape: "
        f"{tuple(evaluation_waveform.shape)}"
    )
    print(
        f"  Reference speech: "
        f"{speech_duration:.2f} seconds"
    )
    print(
        f"  Reference silence: "
        f"{silence_duration:.2f} seconds"
    )
    print(
        f"  Reference overlap: "
        f"{overlap_duration:.2f} seconds"
    )
    print(
        f"  Active speakers: {len(meeting_speakers)}"
    )


total_evaluation_duration = (
    len(MEETING_IDS) * EVALUATION_DURATION
)

print()
print(
    f"Total evaluation audio: "
    f"{total_evaluation_duration / 60:.2f} minutes"
)
print(
    f"Total reference speech: "
    f"{total_reference_speech:.2f} seconds"
)
print(
    f"Total reference silence: "
    f"{total_evaluation_duration - total_reference_speech:.2f} "
    f"seconds"
)
print(
    f"Total reference overlap: "
    f"{total_reference_overlap:.2f} seconds"
)

ES2002d:
  Waveform shape: (4800000,)
  Reference speech: 264.79 seconds
  Reference silence: 35.21 seconds
  Reference overlap: 50.79 seconds
  Active speakers: 4
ES2008d:
  Waveform shape: (4800000,)
  Reference speech: 238.45 seconds
  Reference silence: 61.55 seconds
  Reference overlap: 21.86 seconds
  Active speakers: 4
ES2014d:
  Waveform shape: (4800000,)
  Reference speech: 139.14 seconds
  Reference silence: 160.86 seconds
  Reference overlap: 10.24 seconds
  Active speakers: 4

Total evaluation audio: 15.00 minutes
Total reference speech: 642.38 seconds
Total reference silence: 257.62 seconds
Total reference overlap: 82.89 seconds


## 9. Verify the voice-activity-detection component

As a Phase 1 test, Silero VAD is applied to a separate 30-second excerpt from `ES2002d`, covering 2000 to 2030 seconds. This excerpt is outside the future evaluation interval.

The test only verifies that Silero VAD produces usable speech boundaries. No VAD metric is calculated in Phase 1, and the model's default inference settings are not tuned using the evaluation data.

In [ ]:
def detect_speech_intervals(waveform):
    timestamps = get_speech_timestamps(
        waveform,
        vad_model,
        sampling_rate=TARGET_SAMPLE_RATE,
        return_seconds=False,
    )

    detected_intervals = []

    for timestamp in timestamps:
        start_sample = int(timestamp["start"])
        end_sample = int(timestamp["end"])

        detected_intervals.append(
            {
                "start_sample": start_sample,
                "end_sample": end_sample,
                "start": (
                    start_sample / TARGET_SAMPLE_RATE
                ),
                "end": (
                    end_sample / TARGET_SAMPLE_RATE
                ),
                "duration": (
                    (end_sample - start_sample)
                    / TARGET_SAMPLE_RATE
                ),
            }
        )

    return detected_intervals


SMOKE_TEST_MEETING = "ES2002d"
SMOKE_TEST_START = 2000.0
SMOKE_TEST_END = 2030.0

vad_smoke_waveform = read_audio_segment(
    meeting_id=SMOKE_TEST_MEETING,
    start=SMOKE_TEST_START,
    end=SMOKE_TEST_END,
)

vad_smoke_intervals = detect_speech_intervals(
    vad_smoke_waveform
)

detected_speech_duration = sum(
    interval["duration"]
    for interval in vad_smoke_intervals
)

print("VAD smoke test:")
print(f"Meeting: {SMOKE_TEST_MEETING}")
print(
    f"Meeting-time interval: "
    f"{SMOKE_TEST_START:.0f} to "
    f"{SMOKE_TEST_END:.0f} seconds"
)
print(
    f"Detected speech intervals: "
    f"{len(vad_smoke_intervals)}"
)
print(
    f"Detected speech duration: "
    f"{detected_speech_duration:.2f} seconds"
)

print()
print("First five detected intervals in local excerpt time:")

for interval in vad_smoke_intervals[:5]:
    print(
        f"  {interval['start']:.2f} to "
        f"{interval['end']:.2f} seconds"
    )

VAD smoke test:
Meeting: ES2002d
Meeting-time interval: 2000 to 2030 seconds
Detected speech intervals: 9
Detected speech duration: 26.35 seconds

First five detected intervals in local excerpt time:
  0.77 to 3.58 seconds
  3.87 to 4.57 seconds
  4.83 to 5.76 seconds
  5.86 to 10.46 seconds
  11.36 to 13.92 seconds


## 10. Phase 2 evaluation plan

The two components will be connected serially in Phase 2:

1. Silero VAD will process each continuous 300-second excerpt.
2. Each detected speech interval will be extracted from the waveform.
3. ECAPA-TDNN will create an embedding for that detected interval.
4. Cosine similarity with the 12 enrolled centroids will determine the predicted speaker.

Two metrics will be reported:

- **VAD F1 score:** Reference and predicted speech activity will be compared at 10-millisecond frame resolution. Overlapping speech is still considered speech for this metric. F1 combines the ability to detect reference speech with the ability to avoid false speech detections.
- **Top-1 speaker-identification accuracy:** Accuracy will be calculated only for VAD-detected intervals that overlap speech from exactly one RTTM speaker. A prediction is correct when the highest-scoring centroid belongs to that reference speaker. Intervals containing no reference speech or more than one reference speaker will be analysed separately.

As a simple exploratory overlap heuristic, the difference between the two highest cosine-similarity scores will also be inspected. If this similarity margin is smaller than a threshold that will be selected and frozen before final evaluation, the two highest-scoring speakers will be returned as possible speakers. Otherwise, only the highest-scoring speaker will be returned.

This heuristic is not treated as a reliable overlap detector because ECAPA-TDNN produces a single embedding and was not designed as a multi-speaker classifier. Its behaviour on RTTM-annotated overlap intervals will therefore be examined only as part of the manual error analysis.

No evaluation metrics are calculated in Phase 1.

## 11. Phase 2 analysis plan

The following error categories and data slices will be inspected in Phase 2:

1. **VAD false alarms and missed speech:** Representative false-positive and false-negative regions will be inspected, especially around silence, breaths, hesitations, and low-energy speech.
2. **Short versus long detected intervals:** Eligible single-speaker intervals will be divided using their median duration, and top-1 speaker-identification accuracy will be compared between the short and long groups.
3. **Overlapping or mixed-speaker intervals:** Detected intervals associated with two or more RTTM speakers will be inspected separately. Simultaneous overlap will be distinguished from cases in which one VAD interval merges consecutive speaker turns. For intervals containing exactly two simultaneously active speakers, the analysis will examine whether the similarity-margin heuristic includes both reference speakers among its two candidates and will describe representative successes and failures.

These analyses are planned for Phase 2 and are not performed in Phase 1.